# 03 — Error analysis: false positives, false negatives, and thresholds

Diagnose the trained YOLOv8 detector on the Pascal VOC validation split:

1. Match predictions to ground truth at IoU >= 0.5
2. Rank false positives and false negatives by confidence
3. Analyze class confusion and recall by object size
4. Sweep confidence and NMS IoU thresholds
5. Write `results/error_analysis.md`

This notebook is self-contained: it does not import from `src/`.


In [ ]:
%pip install -q ultralytics pandas matplotlib


## 1. Setup

Expects the dataset from `notebooks/01-data-and-training.ipynb` and a
trained checkpoint in `models/`.


In [ ]:
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "models").exists() and (ROOT.parent / "models").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"
MODELS_DIR = ROOT / "models"
DATASET_YAML = DATA_DIR / "dataset.yaml"
CHECKPOINT = MODELS_DIR / "yolov8n_voc_best.pt"
SPLIT = "val"
CONF = 0.25
IOU = 0.45
IMGSZ = 640
IOU_THRESHOLD = 0.5
SIZE_BUCKETS = (
    (0.0, 0.01, "tiny"),
    (0.01, 0.05, "small"),
    (0.05, 0.2, "medium"),
    (0.2, float("inf"), "large"),
)

print("Project root:", ROOT)
print("Checkpoint:", CHECKPOINT)


## 2. Load the best checkpoint

Falls back to `yolov8n.pt` so the notebook still runs as a smoke test.


In [ ]:
from ultralytics import YOLO

weights = CHECKPOINT if CHECKPOINT.exists() else MODELS_DIR / "yolov8n.pt"
if not weights.exists():
    raise FileNotFoundError(
        "No checkpoint found. Train first with notebook 01 or src/train.py."
    )

model = YOLO(str(weights))
if isinstance(model.names, dict):
    names = [model.names[i] for i in sorted(model.names)]
else:
    names = list(model.names)
print("Weights:", weights)
print("Classes:", len(names))


## 3. Match predictions to ground truth

Predictions are matched greedily by descending confidence at IoU >= 0.5.
A prediction that matches an already-used ground-truth box is a false
positive; any unmatched ground-truth box is a false negative.


In [ ]:
def box_iou(boxes_a, boxes_b):
    if len(boxes_a) == 0 or len(boxes_b) == 0:
        return np.zeros((len(boxes_a), len(boxes_b)), dtype=np.float32)
    top_left = np.maximum(boxes_a[:, None, :2], boxes_b[None, :, :2])
    bottom_right = np.minimum(boxes_a[:, None, 2:], boxes_b[None, :, 2:])
    sizes = np.clip(bottom_right - top_left, 0.0, None)
    intersection = sizes[..., 0] * sizes[..., 1]
    area_a = np.clip(boxes_a[:, 2] - boxes_a[:, 0], 0.0, None)
    area_a *= np.clip(boxes_a[:, 3] - boxes_a[:, 1], 0.0, None)
    area_b = np.clip(boxes_b[:, 2] - boxes_b[:, 0], 0.0, None)
    area_b *= np.clip(boxes_b[:, 3] - boxes_b[:, 1], 0.0, None)
    union = area_a[:, None] + area_b[None, :] - intersection
    return np.where(union > 0.0, intersection / union, 0.0).astype(np.float32)


def load_ground_truth(label_path, image_shape):
    if not label_path.exists() or label_path.stat().st_size == 0:
        return np.zeros((0, 4), dtype=np.float32), np.zeros((0,), dtype=int)
    labels = np.loadtxt(label_path, ndmin=2, dtype=np.float32)
    height, width = image_shape
    scale = np.array([width, height, width, height], dtype=np.float32)
    boxes = labels[:, 1:5] * scale
    xyxy = np.column_stack(
        (
            boxes[:, 0] - boxes[:, 2] / 2.0,
            boxes[:, 1] - boxes[:, 3] / 2.0,
            boxes[:, 0] + boxes[:, 2] / 2.0,
            boxes[:, 1] + boxes[:, 3] / 2.0,
        )
    )
    return xyxy.astype(np.float32), labels[:, 0].astype(int)


def match_detections(
    gt_boxes, gt_classes, pred_boxes, pred_scores, pred_classes
):
    ious = box_iou(gt_boxes, pred_boxes)
    matched_gt, true_positives, false_positives = set(), [], []
    for pred_index in np.argsort(-pred_scores):
        if len(gt_boxes) == 0:
            false_positives.append(int(pred_index))
            continue
        candidate = ious[:, pred_index].copy()
        candidate = np.where(
            gt_classes == pred_classes[pred_index], candidate, 0.0
        )
        if matched_gt:
            candidate[list(matched_gt)] = -1.0
        gt_index = int(np.argmax(candidate))
        if candidate[gt_index] >= IOU_THRESHOLD:
            matched_gt.add(gt_index)
            true_positives.append((gt_index, int(pred_index)))
        else:
            false_positives.append(int(pred_index))
    false_negatives = [
        index for index in range(len(gt_boxes)) if index not in matched_gt
    ]
    return {
        "ious": ious,
        "true_positives": true_positives,
        "false_positives": false_positives,
        "false_negatives": false_negatives,
    }


def size_bucket(area):
    for lower, upper, label in SIZE_BUCKETS:
        if lower <= area < upper:
            return label
    return SIZE_BUCKETS[-1][2]


def normalized_area(box, image_shape):
    height, width = image_shape
    if height <= 0 or width <= 0:
        return 0.0
    area = max(box[2] - box[0], 0.0) * max(box[3] - box[1], 0.0)
    return float(area / (width * height))


## 4. Collect false positives and false negatives

Each false negative records the closest prediction, its confidence, and
its IoU so misses can be ranked even when no box was produced.


In [ ]:
images_dir = DATA_DIR / "images" / SPLIT
labels_dir = DATA_DIR / "labels" / SPLIT
image_paths = sorted(images_dir.glob("*.jpg"))
if not image_paths:
    raise FileNotFoundError(f"No images found in {images_dir}")

false_positives = []
false_negatives = []
confusion = Counter()
buckets = {
    label: {"total": 0, "matched": 0}
    for _, _, label in SIZE_BUCKETS
}
total_gt = total_predictions = total_true_positives = 0

results = model.predict(
    source=[str(path) for path in image_paths],
    conf=CONF,
    iou=IOU,
    imgsz=IMGSZ,
    verbose=False,
    stream=True,
)
for result in results:
    image_path = Path(result.path)
    image_shape = tuple(int(value) for value in result.orig_shape)
    label_path = labels_dir / f"{image_path.stem}.txt"
    gt_boxes, gt_classes = load_ground_truth(label_path, image_shape)

    boxes = getattr(result, "boxes", None)
    if boxes is None or len(boxes) == 0:
        pred_boxes = np.zeros((0, 4), dtype=np.float32)
        pred_scores = np.zeros((0,), dtype=np.float32)
        pred_classes = np.zeros((0,), dtype=int)
    else:
        pred_boxes = boxes.xyxy.cpu().numpy().astype(np.float32)
        pred_scores = boxes.conf.cpu().numpy().astype(np.float32)
        pred_classes = boxes.cls.cpu().numpy().astype(int)

    match = match_detections(
        gt_boxes, gt_classes, pred_boxes, pred_scores, pred_classes
    )
    ious = match["ious"]

    total_gt += len(gt_boxes)
    total_predictions += len(pred_boxes)
    total_true_positives += len(match["true_positives"])

    for gt_index, _ in match["true_positives"]:
        area = normalized_area(gt_boxes[gt_index], image_shape)
        bucket = size_bucket(area)
        buckets[bucket]["total"] += 1
        buckets[bucket]["matched"] += 1

    for gt_index in match["false_negatives"]:
        area = normalized_area(gt_boxes[gt_index], image_shape)
        buckets[size_bucket(area)]["total"] += 1
        best_confidence, best_iou, best_class = 0.0, 0.0, None
        if len(pred_scores) > 0:
            pred_index = int(np.argmax(ious[gt_index]))
            best_iou = float(ious[gt_index, pred_index])
            best_confidence = float(pred_scores[pred_index])
            best_class = names[int(pred_classes[pred_index])]
        class_id = int(gt_classes[gt_index])
        false_negatives.append(
            {
                "image": image_path.name,
                "class_name": names[class_id],
                "confidence": best_confidence,
                "best_iou": best_iou,
                "best_class": best_class,
                "area": area,
            }
        )

    for pred_index in match["false_positives"]:
        class_id = int(pred_classes[pred_index])
        false_positives.append(
            {
                "image": image_path.name,
                "class_name": names[class_id],
                "confidence": float(pred_scores[pred_index]),
                "area": normalized_area(pred_boxes[pred_index], image_shape),
            }
        )

    if len(gt_boxes) > 0 and len(pred_scores) > 0:
        for gt_index, gt_class_id in enumerate(gt_classes):
            pred_index = int(np.argmax(ious[gt_index]))
            if ious[gt_index, pred_index] < IOU_THRESHOLD:
                continue
            actual = names[int(gt_class_id)]
            predicted = names[int(pred_classes[pred_index])]
            confusion[(actual, predicted)] += 1

false_positives.sort(key=lambda item: item["confidence"], reverse=True)
false_negatives.sort(
    key=lambda item: (item["confidence"], -item["area"]), reverse=True
)

precision = (
    total_true_positives / total_predictions if total_predictions else 0.0
)
recall = total_true_positives / total_gt if total_gt else 0.0
summary = pd.Series(
    {
        "images": len(image_paths),
        "ground_truth": total_gt,
        "predictions": total_predictions,
        "true_positives": total_true_positives,
        "false_positives": len(false_positives),
        "false_negatives": len(false_negatives),
        "precision": precision,
        "recall": recall,
    },
    name="value",
)
summary.to_frame()


## 5. Ranked failures


In [ ]:
display(pd.DataFrame(false_positives[:20]))
display(pd.DataFrame(false_negatives[:20]))


## 6. Class confusion and object-size recall

Confusion pairs use the best-overlapping prediction for each ground-truth
box above the IoU threshold, regardless of class. Object-size recall uses
the class-aware matches from section 4.


In [ ]:
confusion_rows = [
    {"actual": actual, "predicted": predicted, "count": count}
    for (actual, predicted), count in confusion.most_common()
]
pd.DataFrame(confusion_rows).head(20)


In [ ]:
size_rows = []
for label, counts in buckets.items():
    total = counts["total"]
    matched = counts["matched"]
    size_rows.append(
        {
            "bucket": label,
            "ground_truth": total,
            "matched": matched,
            "recall": matched / total if total else 0.0,
        }
    )
pd.DataFrame(size_rows)


## 7. Confidence and NMS IoU threshold sweeps

Each value runs a full `model.val()`. Reduce the value lists for a quick
run and increase them for a finer sweep.


In [ ]:
if not DATASET_YAML.exists():
    raise FileNotFoundError(
        "Run notebook 01 or python src/dataset.py first to create "
        "data/dataset.yaml."
    )

CONF_VALUES = [0.10, 0.25, 0.40, 0.55, 0.70]
IOU_VALUES = [0.30, 0.45, 0.60, 0.75]
METRIC_KEYS = {
    "precision": "metrics/precision(B)",
    "recall": "metrics/recall(B)",
    "mAP50": "metrics/mAP50(B)",
    "mAP50-95": "metrics/mAP50-95(B)",
}


def sweep(parameter, values):
    rows = []
    for value in values:
        kwargs = {
            "data": str(DATASET_YAML),
            "imgsz": IMGSZ,
            "batch": 16,
            "plots": False,
            "verbose": False,
            "project": str(RESULTS_DIR / "runs"),
            "name": f"error_sweep_{parameter}_{value:g}",
            "exist_ok": True,
            parameter: value,
        }
        metrics = model.val(**kwargs)
        row = {"value": value}
        row.update(
            {
                name: float(metrics.results_dict[key])
                for name, key in METRIC_KEYS.items()
                if key in metrics.results_dict
            }
        )
        rows.append(row)
    return pd.DataFrame(rows)


conf_sweep = sweep("conf", CONF_VALUES)
iou_sweep = sweep("iou", IOU_VALUES)
display(conf_sweep)
display(iou_sweep)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, frame, parameter in (
    (axes[0], conf_sweep, "conf"),
    (axes[1], iou_sweep, "iou"),
):
    for metric in ("precision", "recall", "mAP50", "mAP50-95"):
        if metric in frame.columns:
            ax.plot(frame["value"], frame[metric], marker="o", label=metric)
    ax.set_xlabel(parameter)
    ax.set_ylabel("Score")
    ax.set_title(f"{parameter} threshold sweep")
    ax.grid(alpha=0.3)
    ax.legend()
fig.tight_layout()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(
    RESULTS_DIR / "threshold_sweeps.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()


In [ ]:
best_conf = float(
    conf_sweep.sort_values("mAP50-95", ascending=False).iloc[0]["value"]
)
best_iou = float(
    iou_sweep.sort_values("mAP50-95", ascending=False).iloc[0]["value"]
)
print("Best confidence:", best_conf)
print("Best IoU:", best_iou)


## 8. Write results/error_analysis.md


In [ ]:
def markdown_table(headers, rows):
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    lines.extend(
        "| " + " | ".join(str(value) for value in row) + " |"
        for row in rows
    )
    return "\n".join(lines)


def score(value):
    return f"{value:.4f}"


def sweep_table(frame, parameter):
    if frame.empty:
        return "Not run."
    best = frame.sort_values("mAP50-95", ascending=False).iloc[0]["value"]
    table = markdown_table(
        [parameter, "precision", "recall", "mAP50", "mAP50-95"],
        [
            [
                f"{row['value']:g}",
                score(row.get("precision", 0.0)),
                score(row.get("recall", 0.0)),
                score(row.get("mAP50", 0.0)),
                score(row.get("mAP50-95", 0.0)),
            ]
            for _, row in frame.iterrows()
        ],
    )
    return f"Best {parameter}: {best:g}\n\n{table}"


lines = [
    "# Error analysis",
    "",
    f"- Weights: `{weights}`",
    f"- Split: `{SPLIT}`",
    f"- Images: {len(image_paths)}",
    "",
    "## Summary",
    "",
    markdown_table(
        ["Metric", "Value"],
        [
            ["Ground-truth boxes", total_gt],
            ["Predictions", total_predictions],
            ["True positives", total_true_positives],
            ["False positives", len(false_positives)],
            ["False negatives", len(false_negatives)],
            ["Precision", score(summary["precision"])],
            ["Recall", score(summary["recall"])],
        ],
    ),
    "",
    "## Top false positives by confidence",
    "",
    (
        markdown_table(
            ["Image", "Class", "Confidence", "Area"],
            [
                [
                    item["image"],
                    item["class_name"],
                    score(item["confidence"]),
                    score(item["area"]),
                ]
                for item in false_positives[:20]
            ],
        )
        if false_positives
        else "None."
    ),
    "",
    "## Top false negatives by confidence",
    "",
    (
        markdown_table(
            [
                "Image",
                "Class",
                "Confidence",
                "Best IoU",
                "Best prediction",
                "Area",
            ],
            [
                [
                    item["image"],
                    item["class_name"],
                    score(item["confidence"]),
                    score(item["best_iou"]),
                    item["best_class"] or "none",
                    score(item["area"]),
                ]
                for item in false_negatives[:20]
            ],
        )
        if false_negatives
        else "None."
    ),
    "",
    "## Class confusion",
    "",
    (
        markdown_table(
            ["Actual", "Predicted", "Count"],
            [
                [row["actual"], row["predicted"], row["count"]]
                for row in confusion_rows[:20]
            ],
        )
        if confusion_rows
        else "None."
    ),
    "",
    "## Recall by object size",
    "",
    markdown_table(
        ["Bucket", "Ground truth", "Matched", "Recall"],
        [
            [
                row["bucket"],
                row["ground_truth"],
                row["matched"],
                score(row["recall"]),
            ]
            for row in size_rows
        ],
    ),
    "",
    "## Confidence sweep",
    "",
    sweep_table(conf_sweep, "conf"),
    "",
    "## IoU sweep",
    "",
    sweep_table(iou_sweep, "iou"),
]
report_path = RESULTS_DIR / "error_analysis.md"
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Saved:", report_path)


## Next steps

- Pick the confidence/IoU pair with the best mAP50-95 for inference.
- Inspect the top false positives and negatives for annotation or domain
  issues.
- Run `python src/error_analysis.py --sweep --weights models/yolov8n_voc_best.pt`.
- Carry the chosen thresholds into the Day 3 inference CLI and the `
  Streamlit app.
